# Asset Pricing Analysis: AI Narrative and Future Stock Returns — Revised Notebook v2

本 notebook 严格按照修正后的研究思路设计：

1. **研究定位**：return predictability / market pricing association，而不是完整的 Fama-French alpha test。
2. **主因变量**：`ret_future_1q`。
3. **补充因变量**：`ret_future_4q`。
4. **主解释变量**：`bert_adoption_dummy`，因为 AI narrative 很稀疏。
5. **扩展解释变量**：`bert_adoption_log_count`、`bert_adoption_per_1k_words`。
6. **固定效应**：
   - 主规格 1：industry FE + quarter FE
   - 主规格 2：firm FE + quarter FE
7. **标准误**：
   - 优先报告 firm-clustered SE 和 two-way clustered SE。
   - 由于季度数较少，two-way clustered SE 只作为谨慎参考。
8. **补充检验**：
   - quarterly cross-sectional regression
   - AI dummy portfolio sort
   - positive-AI intensity sort
   - missingness / attrition diagnostics
   - winsorized robustness
   - factor data availability check

重要说明：本 notebook 不填补 return 缺失值；所有回归均使用 available-case sample。

In [ ]:
# =========================
# Cell 1. Imports and options
# =========================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import statsmodels.api as sm
from scipy import stats

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

print("Packages loaded.")

In [ ]:
# =========================
# Cell 2. Path configuration
# =========================
# 请只修改这里的路径。
# QUARTER_PATH 应指向你的 firm-quarter panel CSV。
# 例如：firm_quarter_panel（2）.csv

BASE = Path("/Users/qinjiayi/Desktop/ds in empirical study/")
QUARTER_PATH = BASE / "firm_quarter_panel（2）.csv"

# 如果你的文件名不同，请在这里改：
# QUARTER_PATH = Path("/your/path/to/firm_quarter_panel.csv")

OUTPUT_DIR = BASE / "asset_pricing_outputs_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("QUARTER_PATH:", QUARTER_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# =========================
# Cell 3. Load data
# =========================

if not QUARTER_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find QUARTER_PATH: {QUARTER_PATH}\n"
        "Please edit Cell 2 and set QUARTER_PATH to your actual firm-quarter panel CSV."
    )

df = pd.read_csv(QUARTER_PATH)

print("Data loaded.")
print("Shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# =========================
# Cell 4. Basic variable cleaning and construction
# =========================

df = df.copy()

# Keep identifiers as strings to avoid dtype problems in clustering and fixed effects.
for col in ["gvkey", "PERMNO"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Ensure year and quarter are numeric.
for col in ["year", "quarter"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Construct firm-quarter identifier.
if {"year", "quarter"}.issubset(df.columns):
    df["yq"] = df["year"].astype("Int64").astype(str) + "Q" + df["quarter"].astype("Int64").astype(str)
else:
    raise ValueError("year and quarter columns are required to construct yq.")

# If gsector exists, use it as industry FE. Otherwise use 2-digit SIC if possible.
if "gsector" in df.columns and df["gsector"].notna().sum() > 0:
    df["industry_fe"] = df["gsector"].astype(str).str.strip()
    INDUSTRY_SOURCE = "gsector"
elif "sich" in df.columns and df["sich"].notna().sum() > 0:
    df["sich_num"] = pd.to_numeric(df["sich"], errors="coerce")
    df["industry_fe"] = (df["sich_num"] // 100).astype("Int64").astype(str)
    INDUSTRY_SOURCE = "2-digit sich"
else:
    df["industry_fe"] = "all"
    INDUSTRY_SOURCE = "none"

# Convert key numeric variables.
numeric_candidates = [
    "ret", "ret_future_1q", "ret_future_4q",
    "size", "leverage", "bm", "profitability", "investment",
    "rd", "capex", "momentum", "mktcap_m",
    "bert_adoption_share", "bert_adoption_count", "bert_adoption_dummy",
    "bert_adoption_log_count", "bert_adoption_per_1k_words",
    "bert_innovation_log_count", "bert_risk_log_count", "bert_hype_log_count",
    "ai_semantic_score", "ai_actionable_share", "ai_speculative_share",
    "lda_topic2_share", "lda_topic8_share"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Ensure AI dummy is 0/1 if present.
if "bert_adoption_dummy" in df.columns:
    df["bert_adoption_dummy"] = (df["bert_adoption_dummy"].fillna(0) > 0).astype(int)

print("Constructed yq and industry_fe.")
print("Industry FE source:", INDUSTRY_SOURCE)
print("Number of quarters:", df["yq"].nunique())
print("Number of firms:", df["gvkey"].nunique() if "gvkey" in df.columns else "gvkey missing")
display(df[["gvkey", "year", "quarter", "yq", "industry_fe"]].head())

In [ ]:
# =========================
# Cell 5. Firm-quarter uniqueness check
# =========================

dup_check = (
    df.groupby(["gvkey", "year", "quarter"], dropna=False)
      .size()
      .reset_index(name="n_rows")
)

print("Firm-quarter duplicate distribution:")
display(dup_check["n_rows"].describe())

max_dup = dup_check["n_rows"].max()
print("Max rows per gvkey-year-quarter:", max_dup)

if max_dup > 1:
    print("WARNING: The data are not strictly firm-quarter unique. You need to aggregate before regression.")
else:
    print("OK: The data are already firm-quarter unique.")

dup_check.to_csv(OUTPUT_DIR / "firm_quarter_uniqueness_check.csv", index=False)

In [ ]:
# =========================
# Cell 6. Missingness and AI sparsity diagnostics
# =========================

diagnostic_vars = [
    "ret_future_1q", "ret_future_4q",
    "bert_adoption_dummy", "bert_adoption_log_count", "bert_adoption_per_1k_words",
    "size", "bm", "leverage", "profitability", "investment", "momentum", "mktcap_m",
    "rd", "capex"
]
diagnostic_vars = [c for c in diagnostic_vars if c in df.columns]

missing_tbl = pd.DataFrame({
    "variable": diagnostic_vars,
    "n_missing": [df[c].isna().sum() for c in diagnostic_vars],
    "missing_rate": [df[c].isna().mean() for c in diagnostic_vars],
    "n_nonmissing": [df[c].notna().sum() for c in diagnostic_vars],
})
display(missing_tbl)
missing_tbl.to_csv(OUTPUT_DIR / "missingness_table.csv", index=False)

ai_vars = [c for c in ["bert_adoption_dummy", "bert_adoption_log_count", "bert_adoption_per_1k_words"] if c in df.columns]
ai_sparsity = []
for c in ai_vars:
    s = df[c]
    ai_sparsity.append({
        "variable": c,
        "mean": s.mean(skipna=True),
        "median": s.median(skipna=True),
        "share_zero": (s.fillna(0) == 0).mean(),
        "share_positive": (s.fillna(0) > 0).mean(),
        "n_positive": int((s.fillna(0) > 0).sum()),
        "n_nonmissing": int(s.notna().sum())
    })
ai_sparsity = pd.DataFrame(ai_sparsity)
display(ai_sparsity)
ai_sparsity.to_csv(OUTPUT_DIR / "ai_sparsity_table.csv", index=False)

In [ ]:
# =========================
# Cell 7. Return-available vs return-missing comparison
# =========================

def compare_available_missing(data, outcome, vars_to_compare):
    out = []
    tmp = data.copy()
    tmp["_return_available"] = tmp[outcome].notna().astype(int)
    for v in vars_to_compare:
        if v not in tmp.columns:
            continue
        g0 = tmp.loc[tmp["_return_available"] == 0, v]
        g1 = tmp.loc[tmp["_return_available"] == 1, v]
        out.append({
            "outcome": outcome,
            "variable": v,
            "mean_available": g1.mean(skipna=True),
            "mean_missing": g0.mean(skipna=True),
            "diff_available_minus_missing": g1.mean(skipna=True) - g0.mean(skipna=True),
            "n_available_nonmissing": g1.notna().sum(),
            "n_missing_nonmissing": g0.notna().sum()
        })
    return pd.DataFrame(out)

compare_vars = [
    "bert_adoption_dummy", "bert_adoption_log_count", "size", "bm", "leverage",
    "profitability", "investment", "momentum", "mktcap_m", "rd", "capex"
]
compare_vars = [c for c in compare_vars if c in df.columns]

ret_miss_1q = compare_available_missing(df, "ret_future_1q", compare_vars)
ret_miss_4q = compare_available_missing(df, "ret_future_4q", compare_vars)

display(ret_miss_1q)
display(ret_miss_4q)

ret_miss_1q.to_csv(OUTPUT_DIR / "return_available_vs_missing_1q.csv", index=False)
ret_miss_4q.to_csv(OUTPUT_DIR / "return_available_vs_missing_4q.csv", index=False)

In [ ]:
# =========================
# Cell 8. Attrition table for model samples
# =========================

MINIMAL_CONTROLS = [c for c in ["size", "bm", "momentum"] if c in df.columns]
BASELINE_CONTROLS = [c for c in ["size", "bm", "leverage", "profitability", "investment", "momentum"] if c in df.columns]
EXTENDED_CONTROLS = [c for c in BASELINE_CONTROLS + ["rd", "capex"] if c in df.columns]

MAIN_AI = "bert_adoption_dummy"
INTENSITY_AI = "bert_adoption_log_count"
STANDARDIZED_AI = "bert_adoption_per_1k_words"

def sample_count(data, outcome, ai_var, controls):
    cols = [outcome, ai_var] + controls + ["gvkey", "yq", "industry_fe"]
    cols = [c for c in cols if c in data.columns]
    tmp = data[cols].dropna()
    return {
        "outcome": outcome,
        "ai_var": ai_var,
        "controls": ", ".join(controls),
        "n_obs": len(tmp),
        "n_firms": tmp["gvkey"].nunique() if "gvkey" in tmp.columns else np.nan,
        "n_quarters": tmp["yq"].nunique() if "yq" in tmp.columns else np.nan,
        "n_industries": tmp["industry_fe"].nunique() if "industry_fe" in tmp.columns else np.nan
    }

attrition_rows = []
for outcome in ["ret_future_1q", "ret_future_4q"]:
    if outcome not in df.columns:
        continue
    for ai in [MAIN_AI, INTENSITY_AI, STANDARDIZED_AI]:
        if ai not in df.columns:
            continue
        for controls in [MINIMAL_CONTROLS, BASELINE_CONTROLS, EXTENDED_CONTROLS]:
            attrition_rows.append(sample_count(df, outcome, ai, controls))

attrition_tbl = pd.DataFrame(attrition_rows)
display(attrition_tbl)
attrition_tbl.to_csv(OUTPUT_DIR / "attrition_by_model_sample.csv", index=False)

In [ ]:
# =========================
# Cell 9. Helper functions: winsorization, FE absorption, and clustered OLS
# =========================

def winsorize_series(s, lower=0.01, upper=0.99):
    # Winsorize a pandas Series without filling missing values.
    s = pd.to_numeric(s, errors="coerce")
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lower=lo, upper=hi)

def _factorize_groups(s):
    # Convert a group Series to integer codes for clustering / fixed effects.
    return pd.factorize(s.astype(str), sort=True)[0]

def absorb_fixed_effects_matrix(data, cols, fe_vars, max_iter=200, tol=1e-10):
    # Residualize columns with respect to one or more fixed effects using alternating projections.
    # This avoids constructing thousands of dummy variables such as C(gvkey).
    Z = data[cols].astype(float).to_numpy(copy=True)
    if not fe_vars:
        return pd.DataFrame(Z, columns=cols, index=data.index)

    groups = [_factorize_groups(data[fe]) for fe in fe_vars]

    for it in range(max_iter):
        Z_old = Z.copy()
        for g in groups:
            tmp = pd.DataFrame(Z)
            tmp["_g"] = g
            means = tmp.groupby("_g").transform("mean").to_numpy()
            Z = Z - means
        max_change = np.nanmax(np.abs(Z - Z_old))
        if max_change < tol:
            break

    return pd.DataFrame(Z, columns=cols, index=data.index)

def run_absorbed_ols(
    data,
    outcome,
    ai_var,
    controls=None,
    fe_vars=None,
    cluster_vars=None,
    label=None,
    min_obs=50,
    min_clusters=2
):
    controls = controls or []
    fe_vars = fe_vars or []
    cluster_vars = cluster_vars or []

    xvars = [ai_var] + controls
    needed = [outcome] + xvars + fe_vars + cluster_vars
    needed = list(dict.fromkeys([c for c in needed if c in data.columns]))
    tmp = data[needed].copy()

    tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    result_rows = []
    base_info = {
        "model": label,
        "outcome": outcome,
        "ai_var": ai_var,
        "controls": ", ".join(controls),
        "fixed_effects": " + ".join(fe_vars) if fe_vars else "none",
        "clusters": " + ".join(cluster_vars) if cluster_vars else "none",
        "n_obs": len(tmp),
        "n_firms": tmp["gvkey"].nunique() if "gvkey" in tmp.columns else np.nan,
        "n_quarters": tmp["yq"].nunique() if "yq" in tmp.columns else np.nan,
        "status": "not_run"
    }

    if len(tmp) < min_obs:
        base_info["status"] = f"too_few_obs_{len(tmp)}"
        return pd.DataFrame([{**base_info, "term": ai_var, "coef": np.nan, "se": np.nan, "t": np.nan, "p": np.nan, "r2": np.nan}])

    usable_x = []
    dropped_x = []
    for x in xvars:
        if x not in tmp.columns:
            dropped_x.append(x)
            continue
        if tmp[x].nunique(dropna=True) <= 1:
            dropped_x.append(x)
        else:
            usable_x.append(x)

    if ai_var not in usable_x:
        base_info["status"] = f"ai_var_no_variation; dropped={dropped_x}"
        return pd.DataFrame([{**base_info, "term": ai_var, "coef": np.nan, "se": np.nan, "t": np.nan, "p": np.nan, "r2": np.nan}])

    resid_cols = [outcome] + usable_x
    try:
        R = absorb_fixed_effects_matrix(tmp, resid_cols, fe_vars)
        y = R[outcome].to_numpy()
        X = R[usable_x]

        final_x = []
        for x in usable_x:
            if np.nanstd(X[x].to_numpy()) > 1e-12:
                final_x.append(x)
        if ai_var not in final_x:
            base_info["status"] = "ai_var_absorbed_by_fixed_effects"
            return pd.DataFrame([{**base_info, "term": ai_var, "coef": np.nan, "se": np.nan, "t": np.nan, "p": np.nan, "r2": np.nan}])

        X = X[final_x].astype(float)
        model = sm.OLS(y, X)

        if cluster_vars:
            cvars = [c for c in cluster_vars if c in tmp.columns]
            if len(cvars) == 1:
                g = _factorize_groups(tmp[cvars[0]])
                if len(np.unique(g)) < min_clusters:
                    fit = model.fit(cov_type="HC1")
                    cov_status = "HC1_fallback_too_few_clusters"
                else:
                    fit = model.fit(cov_type="cluster", cov_kwds={"groups": g, "use_correction": True})
                    cov_status = "clustered"
            elif len(cvars) >= 2:
                g1 = _factorize_groups(tmp[cvars[0]])
                g2 = _factorize_groups(tmp[cvars[1]])
                groups = np.column_stack([g1, g2])
                if len(np.unique(g1)) < min_clusters or len(np.unique(g2)) < min_clusters:
                    fit = model.fit(cov_type="HC1")
                    cov_status = "HC1_fallback_too_few_clusters"
                else:
                    fit = model.fit(cov_type="cluster", cov_kwds={"groups": groups, "use_correction": True})
                    cov_status = "two_way_clustered"
            else:
                fit = model.fit(cov_type="HC1")
                cov_status = "HC1"
        else:
            fit = model.fit(cov_type="HC1")
            cov_status = "HC1"

        base_info["status"] = "ok_" + cov_status
        base_info["r2"] = fit.rsquared

        terms_to_report = [ai_var] + [c for c in controls if c in final_x]
        for term in terms_to_report:
            row = base_info.copy()
            row["term"] = term
            if term in fit.params.index:
                row["coef"] = fit.params[term]
                row["se"] = fit.bse[term]
                row["t"] = fit.tvalues[term]
                row["p"] = fit.pvalues[term]
            else:
                row["coef"] = np.nan
                row["se"] = np.nan
                row["t"] = np.nan
                row["p"] = np.nan
            result_rows.append(row)
        return pd.DataFrame(result_rows)

    except Exception as e:
        base_info["status"] = "error: " + str(e)
        return pd.DataFrame([{**base_info, "term": ai_var, "coef": np.nan, "se": np.nan, "t": np.nan, "p": np.nan, "r2": np.nan}])

def star(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

print("Helper functions defined.")

In [ ]:
# =========================
# Cell 10. Main panel regressions: ret_future_1q
# =========================
# 主模型：ret_future_1q
# 主解释变量：bert_adoption_dummy
# 扩展解释变量：bert_adoption_log_count, bert_adoption_per_1k_words
# FE: industry + quarter; firm + quarter
# SE: firm-clustered and two-way clustered

main_rows = []

OUTCOME_MAIN = "ret_future_1q"

ai_list_main = [c for c in [MAIN_AI, INTENSITY_AI, STANDARDIZED_AI] if c in df.columns]
fe_specs = {
    "industry_quarter_FE": ["industry_fe", "yq"],
    "firm_quarter_FE": ["gvkey", "yq"],
}
cluster_specs = {
    "firm_cluster": ["gvkey"],
    "firm_quarter_two_way_cluster": ["gvkey", "yq"],
}

for ai in ai_list_main:
    for controls_name, controls in [
        ("minimal_controls", MINIMAL_CONTROLS),
        ("baseline_controls", BASELINE_CONTROLS),
    ]:
        for fe_name, fe_vars in fe_specs.items():
            for cl_name, cluster_vars in cluster_specs.items():
                label = f"1Q | {ai} | {controls_name} | {fe_name} | {cl_name}"
                res = run_absorbed_ols(
                    df,
                    outcome=OUTCOME_MAIN,
                    ai_var=ai,
                    controls=controls,
                    fe_vars=fe_vars,
                    cluster_vars=cluster_vars,
                    label=label
                )
                main_rows.append(res)

panel_1q_results = pd.concat(main_rows, ignore_index=True)
display(panel_1q_results[panel_1q_results["term"].isin(ai_list_main)])

panel_1q_results.to_csv(OUTPUT_DIR / "panel_regression_1q_results.csv", index=False)

In [ ]:
# =========================
# Cell 11. Supplementary panel regressions: ret_future_4q
# =========================
# 4Q return 是补充检验，因为：
# 1. 缺失更多；
# 2. future 4Q return 可能有 overlapping return 问题；
# 3. 季度数更少。

supp_rows = []

OUTCOME_SUPP = "ret_future_4q"

if OUTCOME_SUPP in df.columns:
    for ai in ai_list_main:
        for controls_name, controls in [
            ("minimal_controls", MINIMAL_CONTROLS),
            ("baseline_controls", BASELINE_CONTROLS),
        ]:
            for fe_name, fe_vars in fe_specs.items():
                for cl_name, cluster_vars in cluster_specs.items():
                    label = f"4Q | {ai} | {controls_name} | {fe_name} | {cl_name}"
                    res = run_absorbed_ols(
                        df,
                        outcome=OUTCOME_SUPP,
                        ai_var=ai,
                        controls=controls,
                        fe_vars=fe_vars,
                        cluster_vars=cluster_vars,
                        label=label
                    )
                    supp_rows.append(res)

    panel_4q_results = pd.concat(supp_rows, ignore_index=True)
    display(panel_4q_results[panel_4q_results["term"].isin(ai_list_main)])
    panel_4q_results.to_csv(OUTPUT_DIR / "panel_regression_4q_results.csv", index=False)
else:
    print("ret_future_4q not found.")
    panel_4q_results = pd.DataFrame()

In [ ]:
# =========================
# Cell 12. Extended controls: add rd and capex
# =========================
# 目的：
# 检查 AI narrative 的 return relation 是否被 current R&D / capex 吸收。
# 注意：rd 和 capex 可能是机制变量，不能作为主模型控制变量。

extended_rows = []

if len(EXTENDED_CONTROLS) > len(BASELINE_CONTROLS):
    for outcome in [OUTCOME_MAIN, OUTCOME_SUPP]:
        if outcome not in df.columns:
            continue
        for ai in [MAIN_AI, INTENSITY_AI]:
            if ai not in df.columns:
                continue
            for fe_name, fe_vars in fe_specs.items():
                label = f"{outcome} | {ai} | extended_controls | {fe_name} | firm_cluster"
                res = run_absorbed_ols(
                    df,
                    outcome=outcome,
                    ai_var=ai,
                    controls=EXTENDED_CONTROLS,
                    fe_vars=fe_vars,
                    cluster_vars=["gvkey"],
                    label=label
                )
                extended_rows.append(res)

    extended_results = pd.concat(extended_rows, ignore_index=True)
    display(extended_results[extended_results["term"].isin([MAIN_AI, INTENSITY_AI])])
    extended_results.to_csv(OUTPUT_DIR / "panel_regression_extended_controls.csv", index=False)
else:
    print("Extended controls not available because rd/capex are missing or not usable.")
    extended_results = pd.DataFrame()

In [ ]:
# =========================
# Cell 13. Winsorized robustness
# =========================
# 对 return 和连续控制变量做 1% / 99% winsorization。
# AI dummy 不 winsorize。
# 这个检验用于判断结果是否被极端 return 或 accounting variables 驱动。

df_win = df.copy()

winsor_vars = [
    "ret_future_1q", "ret_future_4q",
    "bert_adoption_log_count", "bert_adoption_per_1k_words",
    "size", "bm", "leverage", "profitability", "investment", "momentum", "rd", "capex"
]
winsor_vars = [c for c in winsor_vars if c in df_win.columns]

for c in winsor_vars:
    df_win[c] = winsorize_series(df_win[c], 0.01, 0.99)

winsor_rows = []
for outcome in [OUTCOME_MAIN, OUTCOME_SUPP]:
    if outcome not in df_win.columns:
        continue
    for ai in [MAIN_AI, INTENSITY_AI, STANDARDIZED_AI]:
        if ai not in df_win.columns:
            continue
        for fe_name, fe_vars in fe_specs.items():
            label = f"winsorized | {outcome} | {ai} | baseline_controls | {fe_name} | firm_cluster"
            res = run_absorbed_ols(
                df_win,
                outcome=outcome,
                ai_var=ai,
                controls=BASELINE_CONTROLS,
                fe_vars=fe_vars,
                cluster_vars=["gvkey"],
                label=label
            )
            winsor_rows.append(res)

winsor_results = pd.concat(winsor_rows, ignore_index=True)
display(winsor_results[winsor_results["term"].isin(ai_list_main)])
winsor_results.to_csv(OUTPUT_DIR / "winsorized_panel_regression_results.csv", index=False)

In [ ]:
# =========================
# Cell 14. Quarterly cross-sectional regressions
# =========================
# 每个季度单独做横截面回归：
# future return ~ AI variable + controls + industry dummies
# 然后对季度系数取平均，并计算时间序列 t-stat。
# 注意：季度数少，所以这是补充证据，不是主识别。

def run_quarterly_cs(
    data,
    outcome,
    ai_var,
    controls=None,
    industry_var="industry_fe",
    min_obs=30,
    min_ai_pos=10,
    min_ai_zero=10
):
    controls = controls or []
    rows = []

    for q, g in data.groupby("yq"):
        needed = [outcome, ai_var] + controls + [industry_var]
        needed = [c for c in needed if c in g.columns]
        tmp = g[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()

        n = len(tmp)
        n_pos = int((tmp[ai_var] > 0).sum()) if ai_var in tmp.columns else 0
        n_zero = int((tmp[ai_var] == 0).sum()) if ai_var in tmp.columns else 0

        base = {
            "quarter": q,
            "outcome": outcome,
            "ai_var": ai_var,
            "n_obs": n,
            "n_ai_pos": n_pos,
            "n_ai_zero": n_zero,
            "status": "not_run"
        }

        if n < min_obs:
            rows.append({**base, "status": "too_few_obs", "coef": np.nan, "se": np.nan, "t": np.nan, "p": np.nan, "r2": np.nan})
            continue
        if n_pos < min_ai_pos or n_zero < min_ai_zero:
            rows.append({**base, "status": "insufficient_ai_variation", "coef": np.nan, "se": np.nan, "t": np.nan, "p": np.nan, "r2": np.nan})
            continue

        xcols = [ai_var] + controls
        xcols = [c for c in xcols if c in tmp.columns and tmp[c].nunique(dropna=True) > 1]
        if ai_var not in xcols:
            rows.append({**base, "status": "ai_no_variation", "coef": np.nan, "se": np.nan, "t": np.nan, "p": np.nan, "r2": np.nan})
            continue

        X = tmp[xcols].astype(float)

        if industry_var in tmp.columns and tmp[industry_var].nunique() > 1:
            dummies = pd.get_dummies(tmp[industry_var].astype(str), prefix="ind", drop_first=True, dtype=float)
            X = pd.concat([X.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

        X = sm.add_constant(X, has_constant="add")
        y = tmp[outcome].astype(float).to_numpy()

        try:
            fit = sm.OLS(y, X).fit(cov_type="HC3")
            rows.append({
                **base,
                "status": "ok",
                "coef": fit.params.get(ai_var, np.nan),
                "se": fit.bse.get(ai_var, np.nan),
                "t": fit.tvalues.get(ai_var, np.nan),
                "p": fit.pvalues.get(ai_var, np.nan),
                "r2": fit.rsquared
            })
        except Exception as e:
            rows.append({**base, "status": "error: " + str(e), "coef": np.nan, "se": np.nan, "t": np.nan, "p": np.nan, "r2": np.nan})

    return pd.DataFrame(rows)

def summarize_quarterly_betas(beta_df):
    ok = beta_df.loc[beta_df["status"].eq("ok") & beta_df["coef"].notna()].copy()
    if len(ok) == 0:
        return pd.DataFrame([{
            "outcome": beta_df["outcome"].iloc[0] if len(beta_df) else None,
            "ai_var": beta_df["ai_var"].iloc[0] if len(beta_df) else None,
            "n_quarters_used": 0,
            "mean_beta": np.nan,
            "ts_se": np.nan,
            "ts_t": np.nan,
            "ts_p": np.nan,
            "hac_se": np.nan,
            "hac_t": np.nan,
            "hac_p": np.nan
        }])

    betas = ok["coef"].astype(float).to_numpy()
    T = len(betas)
    mean_beta = np.nanmean(betas)
    ts_se = np.nanstd(betas, ddof=1) / np.sqrt(T) if T > 1 else np.nan
    ts_t = mean_beta / ts_se if ts_se and ts_se > 0 else np.nan
    ts_p = 2 * (1 - stats.t.cdf(abs(ts_t), df=T-1)) if T > 1 and pd.notna(ts_t) else np.nan

    try:
        fit = sm.OLS(ok["coef"].astype(float).to_numpy(), np.ones((T, 1))).fit(
            cov_type="HAC", cov_kwds={"maxlags": min(3, max(0, T - 1))}
        )
        hac_se = float(fit.bse[0])
        hac_t = float(fit.tvalues[0])
        hac_p = float(fit.pvalues[0])
    except Exception:
        hac_se, hac_t, hac_p = np.nan, np.nan, np.nan

    return pd.DataFrame([{
        "outcome": ok["outcome"].iloc[0],
        "ai_var": ok["ai_var"].iloc[0],
        "n_quarters_used": T,
        "mean_beta": mean_beta,
        "ts_se": ts_se,
        "ts_t": ts_t,
        "ts_p": ts_p,
        "hac_se": hac_se,
        "hac_t": hac_t,
        "hac_p": hac_p
    }])

cs_detail_list = []
cs_summary_list = []

for outcome in [OUTCOME_MAIN, OUTCOME_SUPP]:
    if outcome not in df.columns:
        continue
    for ai in [MAIN_AI, INTENSITY_AI, STANDARDIZED_AI]:
        if ai not in df.columns:
            continue
        cs = run_quarterly_cs(
            df,
            outcome=outcome,
            ai_var=ai,
            controls=BASELINE_CONTROLS,
            industry_var="industry_fe",
            min_obs=30,
            min_ai_pos=10,
            min_ai_zero=10
        )
        cs_detail_list.append(cs)
        cs_summary_list.append(summarize_quarterly_betas(cs))

cs_detail = pd.concat(cs_detail_list, ignore_index=True)
cs_summary = pd.concat(cs_summary_list, ignore_index=True)

display(cs_summary)
cs_detail.to_csv(OUTPUT_DIR / "quarterly_cross_sectional_detail.csv", index=False)
cs_summary.to_csv(OUTPUT_DIR / "quarterly_cross_sectional_summary.csv", index=False)

In [ ]:
# =========================
# Cell 15. Portfolio sort: AI dummy portfolio
# =========================
# 每个季度分成 AI adoption dummy = 1 和 = 0 两组。
# 报告 equal-weighted 和 value-weighted High-Low。
# 注意：这里不是 risk-adjusted alpha，只是 raw future return spread。

def weighted_mean(x, w):
    x = pd.to_numeric(x, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    mask = x.notna() & w.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return np.average(x[mask], weights=w[mask])

def portfolio_ai_dummy(
    data,
    outcome,
    ai_dummy="bert_adoption_dummy",
    weight_var="mktcap_m",
    min_pos=10,
    min_zero=10
):
    rows = []
    for q, g in data.groupby("yq"):
        tmp = g[[outcome, ai_dummy, weight_var]].replace([np.inf, -np.inf], np.nan).dropna(subset=[outcome, ai_dummy]).copy()
        n_pos = int((tmp[ai_dummy] == 1).sum())
        n_zero = int((tmp[ai_dummy] == 0).sum())
        base = {"quarter": q, "outcome": outcome, "n_ai": n_pos, "n_non_ai": n_zero}
        if n_pos < min_pos or n_zero < min_zero:
            rows.append({**base, "status": "insufficient_groups", "ew_ai": np.nan, "ew_non_ai": np.nan, "ew_high_low": np.nan,
                         "vw_ai": np.nan, "vw_non_ai": np.nan, "vw_high_low": np.nan})
            continue

        ai_group = tmp[tmp[ai_dummy] == 1]
        non_group = tmp[tmp[ai_dummy] == 0]

        ew_ai = ai_group[outcome].mean()
        ew_non = non_group[outcome].mean()

        if weight_var in tmp.columns:
            vw_ai = weighted_mean(ai_group[outcome], ai_group[weight_var])
            vw_non = weighted_mean(non_group[outcome], non_group[weight_var])
        else:
            vw_ai, vw_non = np.nan, np.nan

        rows.append({
            **base,
            "status": "ok",
            "ew_ai": ew_ai,
            "ew_non_ai": ew_non,
            "ew_high_low": ew_ai - ew_non,
            "vw_ai": vw_ai,
            "vw_non_ai": vw_non,
            "vw_high_low": vw_ai - vw_non if pd.notna(vw_ai) and pd.notna(vw_non) else np.nan
        })
    return pd.DataFrame(rows)

def summarize_portfolio_spread(port_df, spread_col):
    ok = port_df.loc[port_df["status"].eq("ok") & port_df[spread_col].notna()].copy()
    if len(ok) == 0:
        return {
            "spread": spread_col,
            "n_quarters": 0,
            "mean_spread": np.nan,
            "ts_se": np.nan,
            "ts_t": np.nan,
            "ts_p": np.nan,
            "hac_se": np.nan,
            "hac_t": np.nan,
            "hac_p": np.nan
        }
    x = ok[spread_col].astype(float).to_numpy()
    T = len(x)
    mean_x = np.mean(x)
    ts_se = np.std(x, ddof=1) / np.sqrt(T) if T > 1 else np.nan
    ts_t = mean_x / ts_se if pd.notna(ts_se) and ts_se > 0 else np.nan
    ts_p = 2 * (1 - stats.t.cdf(abs(ts_t), df=T-1)) if T > 1 and pd.notna(ts_t) else np.nan

    try:
        fit = sm.OLS(x, np.ones((T, 1))).fit(cov_type="HAC", cov_kwds={"maxlags": min(3, max(0, T - 1))})
        hac_se = float(fit.bse[0])
        hac_t = float(fit.tvalues[0])
        hac_p = float(fit.pvalues[0])
    except Exception:
        hac_se, hac_t, hac_p = np.nan, np.nan, np.nan

    return {
        "spread": spread_col,
        "n_quarters": T,
        "mean_spread": mean_x,
        "ts_se": ts_se,
        "ts_t": ts_t,
        "ts_p": ts_p,
        "hac_se": hac_se,
        "hac_t": hac_t,
        "hac_p": hac_p
    }

port_detail_list = []
port_summary_rows = []

for outcome in [OUTCOME_MAIN, OUTCOME_SUPP]:
    if outcome not in df.columns:
        continue
    port = portfolio_ai_dummy(df, outcome=outcome, ai_dummy=MAIN_AI, weight_var="mktcap_m")
    port_detail_list.append(port)
    for spread_col in ["ew_high_low", "vw_high_low"]:
        row = summarize_portfolio_spread(port, spread_col)
        row["outcome"] = outcome
        row["portfolio"] = "AI dummy: AI minus non-AI"
        port_summary_rows.append(row)

port_detail = pd.concat(port_detail_list, ignore_index=True)
port_summary = pd.DataFrame(port_summary_rows)

display(port_summary)
port_detail.to_csv(OUTPUT_DIR / "portfolio_ai_dummy_detail.csv", index=False)
port_summary.to_csv(OUTPUT_DIR / "portfolio_ai_dummy_summary.csv", index=False)

In [ ]:
# =========================
# Cell 16. Portfolio sort: within-positive AI intensity
# =========================
# 只在 AI-positive observations 中，按照 bert_adoption_log_count 分成 high / low intensity。
# 这是探索性补充，因为 AI-positive 样本本身较少。

def portfolio_positive_intensity(
    data,
    outcome,
    intensity_var="bert_adoption_log_count",
    ai_dummy="bert_adoption_dummy",
    weight_var="mktcap_m",
    min_each_group=10
):
    rows = []
    for q, g in data.groupby("yq"):
        tmp = g.loc[g[ai_dummy] == 1, [outcome, intensity_var, weight_var]].replace([np.inf, -np.inf], np.nan).dropna(subset=[outcome, intensity_var]).copy()
        base = {"quarter": q, "outcome": outcome, "intensity_var": intensity_var, "n_ai_positive": len(tmp)}

        if len(tmp) < 2 * min_each_group or tmp[intensity_var].nunique() < 2:
            rows.append({**base, "status": "insufficient_positive_ai_sample", "ew_high": np.nan, "ew_low": np.nan,
                         "ew_high_low": np.nan, "vw_high": np.nan, "vw_low": np.nan, "vw_high_low": np.nan})
            continue

        med = tmp[intensity_var].median()
        tmp["_high"] = (tmp[intensity_var] > med).astype(int)
        n_high = int((tmp["_high"] == 1).sum())
        n_low = int((tmp["_high"] == 0).sum())
        if n_high < min_each_group or n_low < min_each_group:
            rows.append({**base, "status": "insufficient_high_low_split", "n_high": n_high, "n_low": n_low,
                         "ew_high": np.nan, "ew_low": np.nan, "ew_high_low": np.nan,
                         "vw_high": np.nan, "vw_low": np.nan, "vw_high_low": np.nan})
            continue

        high = tmp[tmp["_high"] == 1]
        low = tmp[tmp["_high"] == 0]

        ew_high = high[outcome].mean()
        ew_low = low[outcome].mean()
        vw_high = weighted_mean(high[outcome], high[weight_var]) if weight_var in high.columns else np.nan
        vw_low = weighted_mean(low[outcome], low[weight_var]) if weight_var in low.columns else np.nan

        rows.append({
            **base,
            "status": "ok",
            "n_high": n_high,
            "n_low": n_low,
            "median_cutoff": med,
            "ew_high": ew_high,
            "ew_low": ew_low,
            "ew_high_low": ew_high - ew_low,
            "vw_high": vw_high,
            "vw_low": vw_low,
            "vw_high_low": vw_high - vw_low if pd.notna(vw_high) and pd.notna(vw_low) else np.nan
        })
    return pd.DataFrame(rows)

pos_detail_list = []
pos_summary_rows = []

for outcome in [OUTCOME_MAIN, OUTCOME_SUPP]:
    if outcome not in df.columns:
        continue
    pos = portfolio_positive_intensity(
        df,
        outcome=outcome,
        intensity_var=INTENSITY_AI,
        ai_dummy=MAIN_AI,
        weight_var="mktcap_m"
    )
    pos_detail_list.append(pos)
    for spread_col in ["ew_high_low", "vw_high_low"]:
        row = summarize_portfolio_spread(pos, spread_col)
        row["outcome"] = outcome
        row["portfolio"] = "Within AI-positive: high intensity minus low intensity"
        pos_summary_rows.append(row)

positive_intensity_detail = pd.concat(pos_detail_list, ignore_index=True)
positive_intensity_summary = pd.DataFrame(pos_summary_rows)

display(positive_intensity_summary)
positive_intensity_detail.to_csv(OUTPUT_DIR / "portfolio_positive_ai_intensity_detail.csv", index=False)
positive_intensity_summary.to_csv(OUTPUT_DIR / "portfolio_positive_ai_intensity_summary.csv", index=False)

In [ ]:
# =========================
# Cell 17. Alternative AI narrative measures robustness
# =========================
# 这些变量不是主变量，只作为 robustness / exploratory analysis。
# 目的：确认结果是否只来自 bert_adoption_dummy，还是其他 AI narrative dimensions 也类似。

alternative_ai_vars = [
    "bert_innovation_log_count",
    "bert_risk_log_count",
    "bert_hype_log_count",
    "ai_semantic_score",
    "ai_actionable_share",
    "ai_speculative_share",
    "lda_topic2_share",
    "lda_topic8_share"
]
alternative_ai_vars = [c for c in alternative_ai_vars if c in df.columns]

alt_rows = []
for outcome in [OUTCOME_MAIN, OUTCOME_SUPP]:
    if outcome not in df.columns:
        continue
    for ai in alternative_ai_vars:
        label = f"alternative_ai | {outcome} | {ai} | baseline_controls | industry_quarter_FE | firm_cluster"
        res = run_absorbed_ols(
            df,
            outcome=outcome,
            ai_var=ai,
            controls=BASELINE_CONTROLS,
            fe_vars=["industry_fe", "yq"],
            cluster_vars=["gvkey"],
            label=label
        )
        alt_rows.append(res)

if alt_rows:
    alternative_ai_results = pd.concat(alt_rows, ignore_index=True)
    display(alternative_ai_results[alternative_ai_results["term"].isin(alternative_ai_vars)])
    alternative_ai_results.to_csv(OUTPUT_DIR / "alternative_ai_measure_results.csv", index=False)
else:
    print("No alternative AI variables found.")
    alternative_ai_results = pd.DataFrame()

In [ ]:
# =========================
# Cell 18. Factor data availability check
# =========================
# 目前这个 notebook 只做 raw return predictability。
# 如果数据中没有 factor columns，不能写 abnormal return / alpha。
# 如果之后补充 Fama-French factors，可以在后续 notebook 中加入 alpha test。

factor_name_candidates = {
    "mktrf": ["mktrf", "mkt_rf", "Mkt-RF", "MKT_RF", "Mkt_RF"],
    "smb": ["smb", "SMB"],
    "hml": ["hml", "HML"],
    "rmw": ["rmw", "RMW"],
    "cma": ["cma", "CMA"],
    "rf": ["rf", "RF"],
    "mom": ["mom", "umd", "MOM", "UMD"]
}

available_factors = {}
for factor, names in factor_name_candidates.items():
    found = [c for c in df.columns if c in names]
    available_factors[factor] = found

factor_check = pd.DataFrame([
    {"factor": k, "available_columns": ", ".join(v), "available": len(v) > 0}
    for k, v in available_factors.items()
])

display(factor_check)
factor_check.to_csv(OUTPUT_DIR / "factor_data_availability_check.csv", index=False)

if not factor_check["available"].any():
    print("No standard Fama-French factor columns found. Current results should be interpreted as return predictability evidence, not alpha evidence.")
else:
    print("Some factor columns appear to be available. Please verify definitions before running factor-adjusted alpha tests.")

In [ ]:
# =========================
# Cell 19. Create compact summary tables for reporting
# =========================

def compact_ai_table(result_df, filename, ai_terms=None):
    if result_df is None or len(result_df) == 0:
        return pd.DataFrame()
    ai_terms = ai_terms or ai_list_main
    tmp = result_df[result_df["term"].isin(ai_terms)].copy()
    if len(tmp) == 0:
        return tmp
    tmp["sig"] = tmp["p"].apply(star)
    tmp["coef_sig"] = tmp.apply(lambda r: f"{r['coef']:.4f}{r['sig']}" if pd.notna(r["coef"]) else "", axis=1)
    keep = [
        "model", "outcome", "term", "coef", "se", "t", "p", "coef_sig",
        "n_obs", "n_firms", "n_quarters", "fixed_effects", "clusters", "r2", "status"
    ]
    keep = [c for c in keep if c in tmp.columns]
    out = tmp[keep].copy()
    out.to_csv(OUTPUT_DIR / filename, index=False)
    return out

compact_1q = compact_ai_table(panel_1q_results, "compact_panel_1q_ai_coefficients.csv")
compact_4q = compact_ai_table(panel_4q_results, "compact_panel_4q_ai_coefficients.csv") if len(panel_4q_results) else pd.DataFrame()
compact_win = compact_ai_table(winsor_results, "compact_winsorized_ai_coefficients.csv")

print("Compact 1Q panel coefficients:")
display(compact_1q.head(30))

print("Compact 4Q panel coefficients:")
display(compact_4q.head(30))

print("Portfolio summary:")
display(port_summary)

print("Cross-sectional summary:")
display(cs_summary)

In [ ]:
# =========================
# Cell 20. Automatic interpretation helper
# =========================
# 这个 cell 不替代人工解释，只帮助快速判断方向。
# 最终论文/PPT要结合前面的样本缺失、稀疏性和模型状态谨慎写。

def interpret_direction(coef, p):
    if pd.isna(coef) or pd.isna(p):
        return "no valid estimate"
    direction = "positive" if coef > 0 else "negative" if coef < 0 else "zero"
    if p < 0.05:
        strength = "statistically significant at 5%"
    elif p < 0.10:
        strength = "marginally significant at 10%"
    else:
        strength = "not statistically significant"
    return f"{direction}, {strength}"

interpret_rows = []

if len(panel_1q_results):
    tmp = panel_1q_results[
        (panel_1q_results["term"] == MAIN_AI) &
        (panel_1q_results["controls"] == ", ".join(BASELINE_CONTROLS)) &
        (panel_1q_results["clusters"] == "gvkey")
    ].copy()
    for _, r in tmp.iterrows():
        interpret_rows.append({
            "source": "panel_1q",
            "model": r["model"],
            "estimate": interpret_direction(r["coef"], r["p"]),
            "coef": r["coef"],
            "p": r["p"],
            "status": r["status"]
        })

if len(panel_4q_results):
    tmp = panel_4q_results[
        (panel_4q_results["term"] == MAIN_AI) &
        (panel_4q_results["controls"] == ", ".join(BASELINE_CONTROLS)) &
        (panel_4q_results["clusters"] == "gvkey")
    ].copy()
    for _, r in tmp.iterrows():
        interpret_rows.append({
            "source": "panel_4q",
            "model": r["model"],
            "estimate": interpret_direction(r["coef"], r["p"]),
            "coef": r["coef"],
            "p": r["p"],
            "status": r["status"]
        })

for _, r in port_summary.iterrows():
    interpret_rows.append({
        "source": "portfolio",
        "model": f"{r['outcome']} | {r['spread']}",
        "estimate": interpret_direction(r["mean_spread"], r["hac_p"]),
        "coef": r["mean_spread"],
        "p": r["hac_p"],
        "status": f"n_quarters={r['n_quarters']}"
    })

interpret_tbl = pd.DataFrame(interpret_rows)
display(interpret_tbl)
interpret_tbl.to_csv(OUTPUT_DIR / "automatic_interpretation_helper.csv", index=False)

In [ ]:
# =========================
# Cell 21. Export all key results into one Excel workbook
# =========================

excel_path = OUTPUT_DIR / "asset_pricing_results_v2.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    missing_tbl.to_excel(writer, sheet_name="missingness", index=False)
    ai_sparsity.to_excel(writer, sheet_name="ai_sparsity", index=False)
    attrition_tbl.to_excel(writer, sheet_name="attrition", index=False)
    ret_miss_1q.to_excel(writer, sheet_name="ret_missing_1q", index=False)
    ret_miss_4q.to_excel(writer, sheet_name="ret_missing_4q", index=False)
    panel_1q_results.to_excel(writer, sheet_name="panel_1q_full", index=False)
    if len(panel_4q_results):
        panel_4q_results.to_excel(writer, sheet_name="panel_4q_full", index=False)
    compact_1q.to_excel(writer, sheet_name="compact_1q", index=False)
    if len(compact_4q):
        compact_4q.to_excel(writer, sheet_name="compact_4q", index=False)
    winsor_results.to_excel(writer, sheet_name="winsorized", index=False)
    cs_summary.to_excel(writer, sheet_name="cs_summary", index=False)
    cs_detail.to_excel(writer, sheet_name="cs_detail", index=False)
    port_summary.to_excel(writer, sheet_name="portfolio_summary", index=False)
    port_detail.to_excel(writer, sheet_name="portfolio_detail", index=False)
    positive_intensity_summary.to_excel(writer, sheet_name="pos_ai_int_summary", index=False)
    positive_intensity_detail.to_excel(writer, sheet_name="pos_ai_int_detail", index=False)
    if len(alternative_ai_results):
        alternative_ai_results.to_excel(writer, sheet_name="alternative_ai", index=False)
    factor_check.to_excel(writer, sheet_name="factor_check", index=False)
    interpret_tbl.to_excel(writer, sheet_name="interpret_helper", index=False)

print("Saved Excel workbook to:", excel_path)
print("Saved individual CSV files to:", OUTPUT_DIR)

## How to interpret this revised notebook

运行完成后，优先看以下输出：

1. `compact_panel_1q_ai_coefficients.csv`  
   - 这是最核心的 1Q return predictability table。
   - 优先看 `bert_adoption_dummy`。
   - 优先看 `baseline_controls`。
   - 同时比较 `industry_fe + quarter_fe` 和 `firm_fe + quarter_fe`。

2. `compact_panel_4q_ai_coefficients.csv`  
   - 这是 4Q supplementary evidence。
   - 不能作为最强结论，因为 4Q return 缺失更多且存在 overlapping return 问题。

3. `portfolio_ai_dummy_summary.csv`  
   - 用于 PPT 展示最直观的 AI vs non-AI future return spread。
   - 但仍然是 raw return spread，不是 factor-adjusted alpha。

4. `quarterly_cross_sectional_summary.csv`  
   - 用于补充传统 asset pricing style 的横截面检验。
   - 由于季度数较少，只能作为 supplementary evidence。

5. `factor_data_availability_check.csv`  
   - 如果没有 factor columns，不能写 abnormal return / alpha。
   - 当前结论应写成 return predictability / market pricing association。

建议写法：

> The asset pricing analysis should be interpreted as a return predictability test rather than a full factor-pricing test. The baseline specification examines whether the presence of AI adoption narratives predicts one-quarter-ahead returns, with industry and quarter fixed effects and a stricter firm and quarter fixed-effects specification. Because AI narratives are sparse, the dummy measure is the primary variable, while intensity-based measures are used as robustness checks.